# RNN Text Classification: Predict the sentiment of IMDB movie reviews

In [111]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
# from google_drive_downloader import GoogleDriveDownloader as gdd
from datasets import load_dataset
from huggingface_hub import login
from sklearn.feature_extraction.text import CountVectorizer
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm, tqdm_notebook

In order to perform deep learning on a GPU (so that everything runs super quick!), CUDA has to be installed and configured. Fortunately, Google Colab already has this set up, but if you want to try this on your own GPU, you can [install CUDA from here](https://developer.nvidia.com/cuda-downloads). Make sure you also [install cuDNN](https://developer.nvidia.com/cudnn) for optimized performance.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
login("HF_TOKEN")
device

device(type='cpu')

## Download the training data

This is a dataset of positive and negative IMDB reviews. We can download the data from a public Google Drive folder.

In [113]:
DATA_PATH = 'data/imdb_reviews.csv'
if not Path(DATA_PATH).is_file():
    # gdd.download_file_from_google_drive(
    #     file_id='1zfM5E6HvKIe7f3rEt1V2gBpw5QOSSKQz',
    #     dest_path=DATA_PATH,
    # )
    dataset = load_dataset("imdb", cache_dir="./data")

In [114]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

## Preprocess the text

In [115]:
# Import necessary libraries
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from torch.utils.data import Dataset

class Sequences(Dataset):
    def __init__(self, dataset, max_seq_len):
        """
        Initializes the Dataset object for sequence data.

        This class handles reading data, creating a vocabulary, encoding text into
        integer sequences, and padding them to a uniform length.

        Args:
            path (str): The file path to the CSV data.
            max_seq_len (int): The fixed length for all sequences after padding.
        """
        self.max_seq_len = max_seq_len

        # TODO: Read the csv file from the given path into a pandas DataFrame.
        df = pd.DataFrame(dataset["train"])

        # --- Vocabulary Creation ---

        # TODO: Initialize a CountVectorizer. ⚙️
        # Use the parameters: stop_words='english', min_df=0.015
        # min_df=0.015 helps filter out very rare words.
        vectorizer = CountVectorizer(stop_words="english", min_df=0.015)

        # TODO: Fit the vectorizer to the 'review' column of the DataFrame to build the vocabulary.
        # Note: We only need to .fit() here, not .fit_transform(), because we will
        # build our own custom encoding logic.
        vectorizer.fit(df["text"], df["label"])


        # TODO: Get the vocabulary from the vectorizer. This is a dict mapping tokens to indices.
        # Hint: It's stored in the 'vocabulary_' attribute after fitting.
        self.token2idx = {token : idx for idx, token in enumerate(vectorizer.vocabulary_)}

        # TODO: Add a special padding token '<PAD>' to your vocabulary. 🔒
        # This token will be used to make all sequences the same length.
        # Its index should be a new, unique value, like one greater than the current max index.
        # Example: self.token2idx['<PAD>'] = max(self.token2idx.values()) + 1
        self.token2idx["<PAD>"] = max(self.token2idx.values()) + 1

        # --- Encoding and Padding Logic ---

        # Get the tokenizer function from the fitted vectorizer
        tokenizer = vectorizer.build_analyzer()

        # TODO: Define a function or lambda to encode a single text sequence. ➡️🔢
        # This function should take a text string (e.g., a review) as input.
        # It should use the `tokenizer` to split the text into tokens.
        # For each token, it should find its index in `self.token2idx`.
        # The function should return a list of these indices.
        # Important: Make sure to only include tokens that are actually in `self.token2idx`.
        self.encode = lambda x: [self.token2idx[token] for token in tokenizer(x) if token in self.token2idx]



        # TODO: Define a function or lambda to pad a single encoded sequence.
        # This function should take a list of token indices (from the encode step) as input.
        # It should append the padding token's index (`self.token2idx['<PAD>']`) to the end
        # of the list until its length equals `self.max_seq_len`.
        # Hint: `padded_seq = original_seq + (self.max_seq_len - len(original_seq)) * [pad_token_index]`
        self.pad = lambda x: x + [self.token2idx["<PAD>"]] * (self.max_seq_len - len(x))


        # --- Apply to the Full Dataset ---

        # TODO: Apply your `self.encode` function to every review in the DataFrame.
        # Also, truncate any sequence that is longer than `self.max_seq_len`.
        # Hint: A list comprehension is great for this:
        # `sequences = [self.encode(review)[:self.max_seq_len] for review in df.review.tolist()]`
        sequences = [self.encode(review)[:self.max_seq_len] for review in df["text"]]

        # TODO: Filter out any empty sequences that might have resulted from the encoding step
        # (e.g., if a review only contained stop words). You must also filter the labels list
        # to keep it aligned with the sequences.
        # A simple for-loop is a good way to do this:
        #
        # final_sequences = []
        # final_labels = []
        # for seq, label in zip(sequences, df.label.tolist()):
        #     if seq:  # check if the sequence is not empty
        #         final_sequences.append(seq)
        #         final_labels.append(label)
        self.lengths = []
        self.labels = [] # Should be the list of final_labels
        self.sequences = [] # Should be the list of final_sequences (before padding)
        for label, review in zip(df["label"], sequences):
            if review:
                self.lengths.append(len(review))
                self.sequences.append(review)
                self.labels.append(label)

        # TODO: Apply your `self.pad` function to every sequence in your filtered list.
        # This is the final list of processed sequences that will be stored.
        self.sequences = [self.pad(seq) for seq in sequences]

    def __getitem__(self, i):
        """
        Gets the i-th sample from the dataset.

        Args:
            i (int): The index of the sample to retrieve.

        Returns:
            tuple: A tuple containing the i-th padded sequence and its label.
        """
        # This assert is a good way to check your padding logic. It ensures every sequence
        # returned by the dataset has the correct length.
        assert len(self.sequences[i]) == self.max_seq_len

        # TODO: Return the i-th sequence and its corresponding label from your final lists.
        return self.sequences[i], self.labels[i], self.lengths[i]

    def __len__(self):
        """
        Returns the total number of samples in the dataset.

        Returns:
            int: The total number of sequences.
        """
        # TODO: Return the total number of sequences in the dataset.
        # Hint: This is the length of your final `self.sequences` list.
        return len(self.sequences)

In [116]:
dataset = Sequences(dataset, max_seq_len=128)

In [117]:
len(dataset.token2idx)

1061

In [118]:
def collate(batch):
    inputs = torch.LongTensor([item[0] for item in batch])
    target = torch.FloatTensor([item[1] for item in batch])
    lengths = torch.tensor([item[2] for item in batch])
    return inputs, target, lengths

batch_size = 2048
train_loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate)

## GRU

![](images/gru_equations.png)

![](images/gru_diagram.png)

In [126]:
# Import necessary libraries
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
class RNN(nn.Module):
    def __init__(
        self,
        vocab_size,
        batch_size,
        embedding_dimension=100,
        hidden_size=128,
        n_layers=1,
        bidirectional=True,
        device='cpu',
    ):
        """
        Initializes the RNN model.

        Args:
            vocab_size (int): The number of unique tokens in the vocabulary.
            batch_size (int): The number of sequences in each batch.
            embedding_dimension (int): The size of the vector for each token.
            hidden_size (int): The number of features in the RNN's hidden state.
            n_layers (int): The number of recurrent layers (for stacking).
            device (str): The device to run the model on ('cpu' or 'cuda').
        """
        super(RNN, self).__init__()
        # --- Store model parameters ---
        self.embedding_dimension = embedding_dimension
        self.vocab_size = vocab_size
        self.n_layers = n_layers
        self.hidden_size = hidden_size
        self.device = device
        self.batch_size = batch_size
        self.bidirectional = bidirectional
        self.n_direction = 2 if self.bidirectional else 1

        # --- Define model layers ---

        # TODO: Define the Embedding layer. 🧠
        # This layer will convert integer sequences (token indices) into dense vector representations.
        # It should map the `vocab_size` to the `embedding_dimension`.
        # Hint: Use nn.Embedding().
        self.encoder = nn.Embedding(self.vocab_size, self.embedding_dimension)

        # TODO: Define the RNN layer. 🔄
        # We will use a GRU (Gated Recurrent Unit) layer.
        # The input size should be the `embedding_dimension` (from the encoder).
        # The hidden size should be `hidden_size`.
        # The number of layers should be `n_layers`.
        # Set `batch_first=True` so that input tensors have the batch dimension first (shape: [batch_size, seq_len, features]).
        # Hint: Use nn.GRU().
        self.rnn = nn.GRU(self.embedding_dimension, self.hidden_size, self.n_layers, batch_first=True, bidirectional=self.bidirectional)

        # TODO: Define the final Decoder (output) layer. 🎯
        # This is a fully connected linear layer that maps the RNN's output to our desired output size.
        # The input size should be `hidden_size`.
        # The output size should be 1 for binary classification.
        # Hint: Use nn.Linear().
        self.decoder = nn.Linear(self.hidden_size * self.n_direction, 1)

    def init_hidden(self):
        """
        Initializes the hidden state for the GRU.
        The hidden state is a tensor of zeros that acts as the initial "memory" for the RNN.

        Returns:
            torch.Tensor: A tensor for the initial hidden state.
        """
        # TODO: Create and return a new tensor for the initial hidden state.
        # The shape should be (self.n_layers, self.batch_size, self.hidden_size).
        # You can initialize it with random values (torch.randn) or zeros (torch.zeros).
        # Make sure to move the tensor to the correct device using `.to(self.device)`.
        return torch.zeros((self.n_layers * self.n_direction, self.batch_size, self.hidden_size), device=self.device)

    def forward(self, inputs, lengths):
        """
        Defines the forward pass of the model.

        Args:
            inputs (torch.Tensor): A batch of padded sequences with shape (batch_size, max_seq_len).

        Returns:
            torch.Tensor: The output logits from the model.
        """
        # This code handles cases where the last batch of data might be smaller than the defined batch_size.
        batch_size = inputs.size(0)
        if batch_size != self.batch_size:
            self.batch_size = batch_size

        # TODO: Pass the input sequences through the embedding layer (self.encoder).
        encoded = self.encoder(inputs)


        #-------------------------------------------------------------------

        pack_encoded = pack_padded_sequence(encoded, torch.tensor(lengths).cpu(), batch_first=True, enforce_sorted=False)

        #-------------------------------------------------------------------

        # TODO: Pass the embedded sequences (`encoded`) and an initial hidden state through the RNN layer (self.rnn).
        # Remember to call `self.init_hidden()` to get the initial hidden state for each forward pass.
        # The RNN will return two things: the `output` from all time steps, and the final `hidden` state.
        output, hidden = self.rnn(pack_encoded, self.init_hidden())

        # TODO: Decode the RNN's output to get the final prediction.
        # For sequence classification, we typically use the output from the very last time step.
        # 1. Select the output from the last time step. The `output` tensor has shape [batch_size, seq_len, hidden_size].
        #    You can get the last time step's output using slicing: `output[:, -1, :]`.
        # 2. Pass this selected output through your decoder layer (self.decoder).
        # 3. Use `.squeeze()` to remove any extra dimensions, resulting in a tensor of shape [batch_size].
        f_dir = hidden.view(self.n_direction, self.n_layers, self.batch_size, self.hidden_size)[0, :, :, :].mean(axis=0).squeeze()
        if self.bidirectional:
            s_dir = hidden.view(self.n_direction, self.n_layers, self.batch_size, self.hidden_size)[1, :, :, :].mean(axis=0).squeeze()
            rnn_out = torch.cat([f_dir, s_dir], dim=-1)
        else:
            rnn_out = f_dir

        # option 2 using output
        # f_dir = output.view(self.batch_size, -1, self.hidden_size, self.n_direction)[:, :, :, 0].squeeze()
        # if self.bidirectional:
        #     s_dir = output.view(self.batch_size, -1, self.hidden_size, self.n_direction)[:, :, :, 1].squeeze()
        #     rnn_out = torch.cat([f_dir, s_dir], dim=-1)
        # else:
        #     rnn_out = f_dir


        output = self.decoder(rnn_out)
        return output

In [128]:
model = RNN(
    hidden_size=128,
    vocab_size=len(dataset.token2idx),
    device=device,
    batch_size=batch_size,
)
model = model.to(device)
model

RNN(
  (encoder): Embedding(1061, 100)
  (rnn): GRU(100, 128, batch_first=True, bidirectional=True)
  (decoder): Linear(in_features=256, out_features=1, bias=True)
)

## Train the model

![](images/rnn_training_diagram.png)

In [127]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)

In [122]:
len(train_loader.dataset)

25000

In [131]:
from tqdm.auto import tqdm
model.train()
train_losses = []
for epoch in range(10):
    progress_bar = tqdm(train_loader, leave=False)
    losses = []
    total = 0
    for inputs, target, lengths in progress_bar:
        inputs, target = inputs.to(device), target.to(device)
        print(inputs.shape, target.shape, lengths.shape )
        model.zero_grad()

        output = model(inputs, lengths).squeeze()

        loss = criterion(output, target)

        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 3)

        optimizer.step()

        progress_bar.set_description(f'Loss: {loss.item():.3f}')

        losses.append(loss.item())
        total += 1

    epoch_loss = sum(losses) / total
    train_losses.append(epoch_loss)

    tqdm.write(f'Epoch #{epoch + 1}\tTrain Loss: {epoch_loss:.3f}')

  0%|          | 0/13 [00:00<?, ?it/s]

torch.Size([2048, 128]) torch.Size([2048]) torch.Size([2048])


/tmp/ipython-input-774104638.py:97: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pack_encoded = pack_padded_sequence(encoded, torch.tensor(lengths).cpu(), batch_first=True, enforce_sorted=False)


torch.Size([2048, 128]) torch.Size([2048]) torch.Size([2048])


KeyboardInterrupt: 

## Analyzing reviews for "Cool Cat Saves the Kids"

![](https://m.media-amazon.com/images/M/MV5BNzE1OTY3OTk5M15BMl5BanBnXkFtZTgwODE0Mjc1NDE@._V1_UY268_CR11,0,182,268_AL_.jpg)

In [ ]:
def predict_sentiment(text):
    model.eval()
    with torch.no_grad():
        test_vector = torch.LongTensor([dataset.pad(dataset.encode(text))]).to(device)

        output = model(test_vector)
        prediction = torch.sigmoid(output).item()

        if prediction > 0.5:
            print(f'{prediction:0.3}: Positive sentiment')
        else:
            print(f'{prediction:0.3}: Negative sentiment')

In [ ]:
test_text = """
This poor excuse for a movie is terrible. It has been 'so good it's bad' for a
while, and the high ratings are a good form of sarcasm, I have to admit. But
now it has to stop. Technically inept, spoon-feeding mundane messages with the
artistic weight of an eighties' commercial, hypocritical to say the least, it
deserves to fall into oblivion. Mr. Derek, I hope you realize you are like that
weird friend that everybody know is lame, but out of kindness and Christian
duty is treated like he's cool or something. That works if you are a good
decent human being, not if you are a horrible arrogant bully like you are. Yes,
Mr. 'Daddy' Derek will end on the history books of the internet for being a
delusional sour old man who thinks to be a good example for kids, but actually
has a poster of Kim Jong-Un in his closet. Destroy this movie if you all have a
conscience, as I hope IHE and all other youtube channel force-closed by Derek
out of SPITE would destroy him in the courts.This poor excuse for a movie is
terrible. It has been 'so good it's bad' for a while, and the high ratings are
a good form of sarcasm, I have to admit. But now it has to stop. Technically
inept, spoon-feeding mundane messages with the artistic weight of an eighties'
commercial, hypocritical to say the least, it deserves to fall into oblivion.
Mr. Derek, I hope you realize you are like that weird friend that everybody
know is lame, but out of kindness and Christian duty is treated like he's cool
or something. That works if you are a good decent human being, not if you are a
horrible arrogant bully like you are. Yes, Mr. 'Daddy' Derek will end on the
history books of the internet for being a delusional sour old man who thinks to
be a good example for kids, but actually has a poster of Kim Jong-Un in his
closet. Destroy this movie if you all have a conscience, as I hope IHE and all
other youtube channel force-closed by Derek out of SPITE would destroy him in
the courts.
"""
predict_sentiment(test_text)

In [ ]:
test_text = """
Cool Cat Saves The Kids is a symbolic masterpiece directed by Derek Savage that
is not only satirical in the way it makes fun of the media and politics, but in
the way in questions as how we humans live life and how society tells us to
live life.

Before I get into those details, I wanna talk about the special effects in this
film. They are ASTONISHING, and it shocks me that Cool Cat Saves The Kids got
snubbed by the Oscars for Best Special Effects. This film makes 2001 look like
garbage, and the directing in this film makes Stanley Kubrick look like the
worst director ever. You know what other film did that? Birdemic: Shock and
Terror. Both of these films are masterpieces, but if I had to choose my
favorite out of the 2, I would have to go with Cool Cat Saves The Kids. It is
now my 10th favorite film of all time.

Now, lets get into the symbolism: So you might be asking yourself, Why is Cool
Cat Orange? Well, I can easily explain. Orange is a color. Orange is also a
fruit, and its a very good fruit. You know what else is good? Good behavior.
What behavior does Cool Cat have? He has good behavior. This cannot be a
coincidence, since cool cat has good behavior in the film.

Now, why is Butch The Bully fat? Well, fat means your wide. You wanna know who
was wide? Hitler. Nuff said this cannot be a coincidence.

Why does Erik Estrada suspect Butch The Bully to be a bully? Well look at it
this way. What color of a shirt was Butchy wearing when he walks into the area?
I don't know, its looks like dark purple/dark blue. Why rhymes with dark? Mark.
Mark is that guy from the Room. The Room is the best movie of all time. What is
the opposite of best? Worst. This is how Erik knew Butch was a bully.

and finally, how come Vivica A. Fox isn't having a successful career after
making Kill Bill.

I actually can't answer that question.

Well thanks for reading my review.
"""
predict_sentiment(test_text)

In [ ]:
test_text = """
Don't let any bullies out there try and shape your judgment on this gem of a
title.

Some people really don't have anything better to do, except trash a great movie
with annoying 1-star votes and spread lies on the Internet about how "dumb"
Cool Cat is.

I wouldn't be surprised to learn if much of the unwarranted negativity hurled
at this movie is coming from people who haven't even watched this movie for
themselves in the first place. Those people are no worse than the Butch the
Bully, the film's repulsive antagonist.

As it just so happens, one of the main points of "Cool Cat Saves the Kids" is
in addressing the attitudes of mean naysayers who try to demean others who
strive to bring good attitudes and fun vibes into people's lives. The message
to be learned here is that if one is friendly and good to others, the world is
friendly and good to one in return, and that is cool. Conversely, if one is
miserable and leaving 1-star votes on IMDb, one is alone and doesn't have any
friends at all. Ain't that the truth?

The world has uncovered a great, new, young filmmaking talent in "Cool Cat"
creator Derek Savage, and I sure hope that this is only the first of many
amazing films and stories that the world has yet to appreciate.

If you are a cool person who likes to have lots of fun, I guarantee that this
is a movie with charm that will uplift your spirits and reaffirm your positive
attitudes towards life.
"""
predict_sentiment(test_text)